In [ ]:
# Core libs
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

# NLTK
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("vader_lexicon", quiet=True)

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data = pd.read_csv("data/freelancer_client_reviews_messy.csv")


In [3]:
data.shape


(150, 2)

In [4]:
len(data)


150

In [5]:
data.columns


Index(['review_text', 'sentiment'], dtype='str')

In [6]:
data.dtypes


review_text      str
sentiment      int64
dtype: object

In [7]:
# Supervised ML
# Labels

data["sentiment"].unique()


array([1, 0])

In [8]:
# Supervised ML

# Classfication - Logistic Regression
# Features (X) - review_text
# Labels (Y) - sentiment

data.head(3)


,review_text,sentiment
0,@mark_designs hired me for a logo redesign and...,1
1,this client changed the brief 11 times in 3 da...,0
2,honestly one of the best clients ive had on up...,1


In [9]:
data["sentiment"].value_counts()


sentiment
1    75
0    75
Name: count, dtype: int64

In [10]:
data.groupby("sentiment").head(5)


,review_text,sentiment
0,@mark_designs hired me for a logo redesign and...,1
1,this client changed the brief 11 times in 3 da...,0
2,honestly one of the best clients ive had on up...,1
3,dude ghosted me after i delivered the final fi...,0
4,@jessica_startup gave me full creative freedom...,1
5,'quick 5 minute task' turned into 3 weeks of r...,0
6,paid me a $200 bonus on top of the agreed rate...,1
7,WARNING to all freelancers: this client will a...,0
8,been working w/ @techbro_ceo for 8 months now ...,1
9,asked me to design a 'simple website' then sen...,0


In [11]:
data[data["sentiment"] == 1].head(5)


,review_text,sentiment
0,@mark_designs hired me for a logo redesign and...,1
2,honestly one of the best clients ive had on up...,1
4,@jessica_startup gave me full creative freedom...,1
6,paid me a $200 bonus on top of the agreed rate...,1
8,been working w/ @techbro_ceo for 8 months now ...,1


In [12]:
data[data["sentiment"] == 0].head(5)


,review_text,sentiment
1,this client changed the brief 11 times in 3 da...,0
3,dude ghosted me after i delivered the final fi...,0
5,'quick 5 minute task' turned into 3 weeks of r...,0
7,WARNING to all freelancers: this client will a...,0
9,asked me to design a 'simple website' then sen...,0


## Phase 2: NLP TEXT PREPROCESSING PIPELINE


In [13]:
# Sample texts to test against
sample_text = [
    "@mark_designs hired me for a logo redesign here is Dummy http://www.upwork.com PAID WITHIN 2 hours 🙌🙌 dream client $100 fr",
    "WARNING!!! This client will ALWAYS ask for revisions... never pays on time #scam @badclient",
    "Honestly one of the BEST clients I've had on Upwork!! Paid $500 bonus :) #grateful #freelance",
    "RT @techbro_ceo: been working w/ this guy for 8 months, 100% recommend! https://t.co/xyz123 😊",
]


In [14]:
# --- Individual Preprocessing Functions ---

# Contraction map — expand BEFORE removing punctuation so "don't" → "do not" (not "dont")
CONTRACTIONS = {
    "won't": "will not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "mustn't": "must not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "hasn't": "has not",
    "haven't": "have not",
    "hadn't": "had not",
    "doesn't": "does not",
    "don't": "do not",
    "didn't": "did not",
    "needn't": "need not",
    "i'm": "i am",
    "i've": "i have",
    "i'll": "i will",
    "i'd": "i would",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "they're": "they are",
    "they've": "they have",
    "they'll": "they will",
    "we're": "we are",
    "we've": "we have",
    "you're": "you are",
    "you've": "you have",
    "he's": "he is",
    "she's": "she is",
}


def expand_contractions(text):
    """Expand contractions so negations survive punctuation removal.
    e.g. don't -> do not, won't -> will not, isn't -> is not
    """
    for contraction, expansion in CONTRACTIONS.items():
        text = re.sub(re.escape(contraction), expansion, text, flags=re.IGNORECASE)
    return text


def to_lowercase(text):
    """Convert text to lowercase."""
    return text.lower()


def remove_urls(text):
    """Remove http/https URLs and www links."""
    return re.sub(r"https?://\S+|www\.\S+", "", text)


def remove_mentions(text):
    """Remove @username mentions."""
    return re.sub(r"@\w+", "", text)


def remove_hashtags(text):
    """Strip the # symbol but keep the word."""
    return re.sub(r"#", "", text)


def remove_emojis(text):
    """Remove emoji / unicode symbol characters."""
    emoji_pattern = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002500-\U00002bff"
        "\U00002702-\U000027b0"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)


def remove_numbers(text):
    """Remove numeric digits."""
    return re.sub(r"\d+", "", text)


def remove_punctuation(text):
    """Remove punctuation and special characters."""
    return re.sub(r"[^\w\s]", "", text)


def remove_extra_spaces(text):
    """Collapse multiple whitespace into a single space."""
    return re.sub(r"\s+", " ", text).strip()


def remove_short_tokens(tokens, min_len=2):
    """Drop single-character noise tokens like 'w', 'u', 'r', 'fr'."""
    return [t for t in tokens if len(t) >= min_len]


In [15]:
# Negation words that must be kept — they carry critical negative sentiment
NEGATION_WORDS = {"no", "not", "nor", "never", "neither", "against", "without"}


def remove_stopwords(text):
    """Remove stop words but KEEP negation words (no, not, never, etc.)."""
    base_stopwords = set(stopwords.words("english"))
    custom_stopwords = base_stopwords - NEGATION_WORDS
    return " ".join([word for word in text.split() if word not in custom_stopwords])


def tokenize_text(text):
    """Split text into a list of word tokens."""
    return text.split()


def _get_wordnet_pos(word):
    """Map NLTK POS tag to WordNet POS for accurate lemmatization."""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)  # default to NOUN


def lemmatize_tokens(tokens):
    """Lemmatize tokens using POS-aware WordNetLemmatizer.
    Returns real dictionary words (e.g. 'revisions' -> 'revision',
    'always' -> 'always') instead of broken stems like 'revis', 'alway'.
    """
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token, _get_wordnet_pos(token)) for token in tokens]


In [16]:
# --- Full Preprocessing Pipeline ---


def preprocess_pipeline(text):
    """
    Improved preprocessing pipeline.

    Steps:
      1.  expand_contractions  <- NEW: don't→do not, won't→will not (before punctuation removal!)
      2.  to_lowercase
      3.  remove_urls
      4.  remove_mentions
      5.  remove_hashtags
      6.  remove_emojis
      7.  remove_numbers
      8.  remove_punctuation
      9.  remove_extra_spaces
      10. remove_stopwords     (keeps negation words: no, not, never ...)
      11. tokenize_text
      12. lemmatize_tokens     (real words, not broken stems)
      13. remove_short_tokens  <- NEW: drops noise like 'w', 'fr', 'u'
    """
    text = expand_contractions(
        text
    )  # must come first — before punctuation strips apostrophes
    text = to_lowercase(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)
    text = remove_stopwords(text)
    tokens = tokenize_text(text)
    tokens = lemmatize_tokens(tokens)
    tokens = remove_short_tokens(tokens)
    return " ".join(tokens)


## Phase 3: Testing Each Function Step-by-Step


In [17]:
print("=== Original Samples ===")
for i, s in enumerate(sample_text, 1):
    print(f"[{i}] {s}")


=== Original Samples ===
[1] @mark_designs hired me for a logo redesign here is Dummy http://www.upwork.com PAID WITHIN 2 hours 🙌🙌 dream client $100 fr
[2] WARNING!!! This client will ALWAYS ask for revisions... never pays on time #scam @badclient
[3] Honestly one of the BEST clients I've had on Upwork!! Paid $500 bonus :) #grateful #freelance
[4] RT @techbro_ceo: been working w/ this guy for 8 months, 100% recommend! https://t.co/xyz123 😊


In [18]:
# Test each function step-by-step on the second sample (has contractions + negation)
test = "This client won't pay and doesn't respond. They always ask for revisions. Never again!"
print(f"Original           : {test}")

test = expand_contractions(test)
print(f"Contractions       : {test}")

test = to_lowercase(test)
print(f"Lowercase          : {test}")

test = remove_urls(test)
print(f"No URLs            : {test}")

test = remove_mentions(test)
print(f"No Mentions        : {test}")

test = remove_hashtags(test)
print(f"No Hashtags        : {test}")

test = remove_emojis(test)
print(f"No Emojis          : {test}")

test = remove_numbers(test)
print(f"No Numbers         : {test}")

test = remove_punctuation(test)
print(f"No Punctuation     : {test}")

test = remove_extra_spaces(test)
print(f"Clean Spaces       : {test}")

test = remove_stopwords(test)
print(f"No Stopwords       : {test}")

tokens = tokenize_text(test)
print(f"Tokens             : {tokens}")

tokens = lemmatize_tokens(tokens)
print(f"Lemmatized         : {tokens}")

tokens = remove_short_tokens(tokens)
print(f"No Short Tokens    : {tokens}")


Original           : This client won't pay and doesn't respond. They always ask for revisions. Never again!
Contractions       : This client will not pay and does not respond. They always ask for revisions. Never again!
Lowercase          : this client will not pay and does not respond. they always ask for revisions. never again!
No URLs            : this client will not pay and does not respond. they always ask for revisions. never again!
No Mentions        : this client will not pay and does not respond. they always ask for revisions. never again!
No Hashtags        : this client will not pay and does not respond. they always ask for revisions. never again!
No Emojis          : this client will not pay and does not respond. they always ask for revisions. never again!
No Numbers         : this client will not pay and does not respond. they always ask for revisions. never again!
No Punctuation     : this client will not pay and does not respond they always ask for revisions never again

## Phase 4: Run Full Pipeline on All Samples & Dataset


In [19]:
# Test pipeline on all sample texts
print("=== Pipeline Output ===")
for i, s in enumerate(sample_text, 1):
    result = preprocess_pipeline(s)
    print(f"[{i}] {result}")


=== Pipeline Output ===
[1] hire logo redesign dummy paid within hour dream client fr
[2] warn client always ask revision never pay time scam
[3] honestly one best client upwork paid bonus grateful freelance
[4] rt work guy month recommend


In [20]:
# # Apply pipeline to the full dataset
# data["tokens"] = data["review_text"].apply(preprocess_pipeline)

# # Preview results
# data[["review_text", "tokens", "sentiment"]].head(5)


In [ ]:
# Apply preprocessing pipeline
data["clean_text"] = data["review_text"].apply(preprocess_pipeline)

# VADER on ORIGINAL text (VADER is calibrated for raw social text — don't preprocess it)
sia = SentimentIntensityAnalyzer()
vader_scores = data["review_text"].apply(lambda t: sia.polarity_scores(t))

data["vader_neg"] = vader_scores.apply(lambda s: s["neg"])
data["vader_pos"] = vader_scores.apply(lambda s: s["pos"])
data["vader_compound"] = vader_scores.apply(lambda s: s["compound"])

data[["review_text", "clean_text", "vader_neg", "vader_pos", "vader_compound"]].head(5)


,review_text,clean_text,vader_neg,vader_pos,vader_compound
0,@mark_designs hired me for a logo redesign and...,hire logo redesign paid within hour delivery d...,0.000,0.118,0.2500
1,this client changed the brief 11 times in 3 da...,client change brief time day ask discount bc t...,0.000,0.000,0.0000
2,honestly one of the best clients ive had on up...,honestly one best client ive upwork clear brie...,0.000,0.502,0.9022
3,dude ghosted me after i delivered the final fi...,dude ghost deliver final file no payment no re...,0.351,0.000,-0.7200
4,@jessica_startup gave me full creative freedom...,give full creative freedom brand identity proj...,0.000,0.439,0.9141


In [22]:
data[["review_text", "clean_text"]].head(10)


,review_text,clean_text
0,@mark_designs hired me for a logo redesign and...,hire logo redesign paid within hour delivery d...
1,this client changed the brief 11 times in 3 da...,client change brief time day ask discount bc t...
2,honestly one of the best clients ive had on up...,honestly one best client ive upwork clear brie...
3,dude ghosted me after i delivered the final fi...,dude ghost deliver final file no payment no re...
4,@jessica_startup gave me full creative freedom...,give full creative freedom brand identity proj...
5,'quick 5 minute task' turned into 3 weeks of r...,quick minute task turn week revision zero extr...
6,paid me a $200 bonus on top of the agreed rate...,paid bonus top agree rate bc say exceed expect...
7,WARNING to all freelancers: this client will a...,warn freelancer client ask test project disapp...
8,been working w/ @techbro_ceo for 8 months now ...,work month every single invoice paid time that...
9,asked me to design a 'simple website' then sen...,ask design simple website sent page requiremen...


In [23]:
data.columns


Index(['review_text', 'sentiment', 'clean_text', 'vader_neg', 'vader_pos',
       'vader_compound'],
      dtype='str')

In [24]:
X = data["clean_text"]  # Feature(s)
y = data["sentiment"]  # Target or Labels


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # stratify = equal class balance in both splits
)


In [26]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),  # unigrams + bigrams: captures "never pay", "ask revision"
    max_features=3000,  # limit vocab to avoid overfitting on tiny dataset
    sublinear_tf=True,  # apply log(tf) — reduces dominance of frequent terms
    min_df=2,  # ignore terms that appear in fewer than 2 documents
)


In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# VADER features aligned to the same train/test split via index
vader_cols = ["vader_neg", "vader_pos", "vader_compound"]
X_train_vader = csr_matrix(data.loc[X_train.index, vader_cols].values)
X_test_vader = csr_matrix(data.loc[X_test.index, vader_cols].values)

# Combine TF-IDF + VADER into one feature matrix
X_train_combined = hstack([X_train_tfidf, X_train_vader])
X_test_combined = hstack([X_test_tfidf, X_test_vader])

print("Train features:", X_train_combined.shape)
print("Test  features:", X_test_combined.shape)


Train features: (120, 240)
Test  features: (30, 240)


In [ ]:
model = LogisticRegression(
    max_iter=1000, C=1.0, class_weight="balanced", random_state=42
)


In [29]:
model.fit(X_train_combined, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [30]:
y_pred = model.predict(X_test_combined)


In [31]:
accuracy = accuracy_score(y_test, y_pred)


In [32]:
accuracy


0.8666666666666667

In [33]:
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


              precision    recall  f1-score   support

    Negative       0.92      0.80      0.86        15
    Positive       0.82      0.93      0.88        15

    accuracy                           0.87        30
   macro avg       0.87      0.87      0.87        30
weighted avg       0.87      0.87      0.87        30



In [34]:
# Compare models with 5-fold cross-validation on the full dataset
tfidf_settings = dict(
    ngram_range=(1, 2), max_features=3000, sublinear_tf=True, min_df=2
)

models = {
    "ComplementNB (current)": ComplementNB(),
    "LogisticRegression": LogisticRegression(
        max_iter=1000, C=1.0, class_weight="balanced", random_state=42
    ),
}

print(f"{'Model':<30} {'Mean Accuracy':>14}  {'Std':>6}")
print("-" * 55)
for name, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(**tfidf_settings)), ("clf", clf)])
    scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    print(f"{name:<30} {scores.mean():.3f}          ±{scores.std():.3f}")


Model                           Mean Accuracy     Std
-------------------------------------------------------
ComplementNB (current)         0.807          ±0.061
LogisticRegression             0.773          ±0.057


In [35]:
brand_new_review = """
this is the worst #client ever. they never pay on time and always ask for revisions. #scam @badclient
"""


In [36]:
clean_review = preprocess_pipeline(brand_new_review)


In [37]:
clean_review


'bad client ever never pay time always ask revision scam'

In [ ]:
tfidf_review = tfidf.transform([clean_review])

# VADER on original review text
vader_review = sia.polarity_scores(brand_new_review)
vader_feats = csr_matrix(
    [[vader_review["neg"], vader_review["pos"], vader_review["compound"]]]
)

review_vector = hstack([tfidf_review, vader_feats])


In [39]:
review_prediction = model.predict(review_vector)


In [40]:
if review_prediction == 1:
    print("Predicted Sentiment: Positive")
else:
    print("Predicted Sentiment: Negative")


Predicted Sentiment: Negative
